# Data Profiling and Quality Risk Analysis (Workshop §6.3)

**Purpose:** gather evidence about both sources *before* defining cleaning or validation rules. This notebook **only reads** data: nothing is modified or written back to the sources.

**Reproducibility:** run from the repository root (`jupyter nbconvert --to notebook --execute notebooks/data_profiling.ipynb`) or interactively. Every evidence table is also saved as CSV in `docs/evidence/profiling/` with an ID (`P-S-xx` Spotify, `P-G-xx` Grammy, `P-X-xx` cross-source) that is referenced from the risk table in `README.md` (§5).

**Sources**
- Spotify: `data/raw/spotify_dataset.csv` (file source, as the pipeline will read it).
- Grammy: the relational **source database** table `grammy_awards` (the operational source the pipeline extracts from). If the database is not reachable the notebook falls back to the preserved CSV and says so explicitly.

Requirements being protected: **AR-01** (popularity of Grammy vs non-Grammy artists), **AR-02** (genre concentration of Grammy wins), **AR-03** (audio profile of Grammy tracks by decade). See `README.md` (§2).

In [1]:
import os, re, unicodedata, warnings
from pathlib import Path

import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 70)

# --- locate the repository root whether run from root or from notebooks/ ---
ROOT = Path.cwd()
if not (ROOT / "data" / "raw").exists():
    ROOT = ROOT.parent
EVID = ROOT / "docs" / "evidence" / "profiling"
EVID.mkdir(parents=True, exist_ok=True)
FIG = EVID

# optional .env (local runs); inside Docker the variables already exist
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except Exception:
    pass

def save(df, evidence_id, name, index=True):
    """Persist an evidence table as CSV so every reported risk is backed by a file, not a screenshot."""
    path = EVID / f"{evidence_id}_{name}.csv"
    df.to_csv(path, index=index)
    return path.name

SPOTIFY_CSV = ROOT / "data" / "raw" / "spotify_dataset.csv"
GRAMMY_CSV = ROOT / "data" / "raw" / "the_grammy_awards.csv"

spotify = pd.read_csv(SPOTIFY_CSV)
spotify = spotify.rename(columns={"Unnamed: 0": "csv_index"})   # naming only: profiling view, values untouched

grammy_origin = "CSV fallback (database not reachable)"
try:
    import psycopg2
    conn = psycopg2.connect(host=os.environ["DATA_DB_HOST"], port=int(os.environ["DATA_DB_PORT"]),
                            user=os.environ["DATA_DB_USER"], password=os.environ["DATA_DB_PASSWORD"],
                            dbname=os.environ["SOURCE_DB_NAME"])
    grammy = pd.read_sql("SELECT * FROM grammy_awards ORDER BY grammy_row_id", conn)
    conn.close()
    grammy_origin = "relational source database (table grammy_awards)"
except Exception as exc:
    grammy = pd.read_csv(GRAMMY_CSV)
    print("DB not used:", type(exc).__name__, str(exc)[:80])

print("Spotify  :", spotify.shape, "from", SPOTIFY_CSV.name)
print("Grammy   :", grammy.shape, "from", grammy_origin)
pd.Series({"grammy_origin": grammy_origin, "grammy_rows": len(grammy), "spotify_rows": len(spotify)}).to_frame("value").to_csv(EVID / "P-run_context.csv")

Spotify  : (114000, 21) from spotify_dataset.csv
Grammy   : (4810, 11) from relational source database (table grammy_awards)


## 1 · Structure (schema, columns, row counts, data types, fields required by the requirements)

In [2]:
def structure(df, required):
    out = pd.DataFrame({"dtype": df.dtypes.astype(str), "non_null": df.notna().sum(), "distinct": df.nunique()})
    out["required_by"] = [", ".join(required.get(c, [])) or "-" for c in out.index]
    return out

req_spotify = {
    "track_id": ["AR-01","AR-02","AR-03"], "artists": ["AR-01","AR-02","AR-03"], "track_name": ["AR-03"],
    "album_name": ["AR-03"], "popularity": ["AR-01"], "track_genre": ["AR-02"],
    "danceability": ["AR-03"], "energy": ["AR-03"], "valence": ["AR-03"], "loudness": ["AR-03"],
    "tempo": ["AR-03"], "duration_ms": ["AR-03"], "explicit": ["AR-03"],
}
req_grammy = {
    "year": ["AR-01","AR-02","AR-03"], "category": ["AR-02","AR-03"], "nominee": ["AR-03"],
    "artist": ["AR-01","AR-02","AR-03"], "winner": ["AR-01"],
}
p_s_01 = structure(spotify, req_spotify); p_g_01 = structure(grammy, req_grammy)
print("P-S-01 Spotify structure"); display(p_s_01)
print("P-G-01 Grammy structure");  display(p_g_01)
save(p_s_01, "P-S-01", "structure"); save(p_g_01, "P-G-01", "structure")

P-S-01 Spotify structure


,dtype,non_null,distinct,required_by
csv_index,int64,114000,114000,-
track_id,str,114000,89741,"AR-01, AR-02, AR-03"
artists,str,113999,31437,"AR-01, AR-02, AR-03"
album_name,str,113999,46589,AR-03
track_name,str,113999,73608,AR-03
popularity,int64,114000,101,AR-01
duration_ms,int64,114000,50697,AR-03
explicit,bool,114000,2,AR-03
danceability,float64,114000,1174,AR-03
energy,float64,114000,2083,AR-03


P-G-01 Grammy structure


,dtype,non_null,distinct,required_by
grammy_row_id,int64,4810,4810,-
year,int64,4810,62,"AR-01, AR-02, AR-03"
title,str,4810,62,-
published_at,"datetime64[us, UTC]",4810,4,-
updated_at,"datetime64[us, UTC]",4810,10,-
category,str,4810,638,"AR-02, AR-03"
nominee,str,4804,4131,AR-03
artist,str,2970,1658,"AR-01, AR-02, AR-03"
workers,str,2620,2366,-
img,str,3443,1463,-


'P-G-01_structure.csv'

## 2 · Completeness (missing values, interpreted in context)

In [3]:
def missing(df):
    m = pd.DataFrame({"missing": df.isna().sum(), "missing_pct": (df.isna().mean() * 100).round(2)})
    return m[m.missing > 0]

p_s_02 = missing(spotify); p_g_02 = missing(grammy)
print("P-S-02 Spotify missing"); display(p_s_02)
print("P-G-02 Grammy missing");  display(p_g_02)
save(p_s_02, "P-S-02", "missing"); save(p_g_02, "P-G-02", "missing")

print("\nSpotify rows with any missing value:")
display(spotify[spotify.isna().any(axis=1)][["csv_index","track_id","artists","album_name","track_name","duration_ms"]])

# Context: WHERE is Grammy 'artist' missing? (a column that is 38% empty is not random noise)
g = grammy.copy()
by_cat = (g.assign(artist_missing=g.artist.isna())
            .groupby("category").agg(rows=("artist_missing","size"), artist_missing=("artist_missing","sum")))
by_cat["artist_missing_pct"] = (by_cat.artist_missing / by_cat.rows * 100).round(1)
p_g_08 = by_cat[by_cat.artist_missing > 0].sort_values("artist_missing", ascending=False)
print("P-G-08 Grammy: categories with missing artist (top 12)"); display(p_g_08.head(12))
save(p_g_08, "P-G-08", "artist_missing_by_category")

sotY = g[g.category.str.lower() == "song of the year"]
print(f"\nSong Of The Year rows: {len(sotY)}; with artist missing: {sotY.artist.isna().sum()}")
print("Rows with artist missing but workers present:", int((g.artist.isna() & g.workers.notna()).sum()),
      "| both missing:", int((g.artist.isna() & g.workers.isna()).sum()))
print("Rows with missing nominee:"); display(g[g.nominee.isna()][["year","category","nominee","artist"]])

# Can the missing artist be recovered? (an observation about the data, NOT a transformation applied here)
miss = g[g.artist.isna() & g.workers.notna()]
paren = miss.workers.str.extract(r"\(([^()]+)\)\s*$")[0]
bna = g[g.category.str.lower() == "best new artist"]
p_g_09 = pd.Series({
    "missing_artist_rows_total": int(g.artist.isna().sum()),
    "...of_which_workers_present": len(miss),
    "...workers_ends_with_(parenthesized_name)": int(paren.notna().sum()),
    "...parenthesized_pct_of_workers_present": round(paren.notna().mean() * 100, 1),
    "best_new_artist_rows": len(bna),
    "best_new_artist_artist_missing": int(bna.artist.isna().sum()),
    "best_new_artist_nominee_is_the_artist_(example)": bna.nominee.iloc[0],
}).to_frame("value")
print("P-G-09 Possible recovery paths for missing Grammy artist"); display(p_g_09); save(p_g_09, "P-G-09", "artist_recovery_paths")
display(miss[paren.notna()][["category","nominee","workers"]].head(4))

P-S-02 Spotify missing


,missing,missing_pct
artists,1,0.0
album_name,1,0.0
track_name,1,0.0


P-G-02 Grammy missing


,missing,missing_pct
nominee,6,0.12
artist,1840,38.25
workers,2190,45.53
img,1367,28.42



Spotify rows with any missing value:


...


Song Of The Year rows: 70; with artist missing: 70
Rows with artist missing but workers present: 1654 | both missing: 186
Rows with missing nominee:


,year,category,nominee,artist
2261,2000,"Remixer of the Year, Non-Classical",NaN,NaN
2359,1999,"Remixer Of The Year, Non-Classical",NaN,NaN
2454,1998,"Remixer Of The Year, Non-classical",NaN,NaN
2547,1997,"Remixer Of The Year, Non-Classical",NaN,NaN
4525,1965,Best New Country & Western Artist,NaN,NaN
4573,1964,Best New Country & Western Artist Of 1964,NaN,NaN


P-G-09 Possible recovery paths for missing Grammy artist


,value
missing_artist_rows_total,1840
...of_which_workers_present,1654
...workers_ends_with_(parenthesized_name),1333
...parenthesized_pct_of_workers_present,80.6
best_new_artist_rows,51
best_new_artist_artist_missing,50
best_new_artist_nominee_is_the_artist_(example),Billie Eilish


,category,nominee,workers
16,Song Of The Year,Bad Guy,"Billie Eilish O'Connell & Finneas O'Connell, songwriters (Billie E..."
17,Song Of The Year,Always Remember Us This Way,"Natalie Hemby, Lady Gaga, Hillary Lindsey & Lori McKenna, songwrit..."
18,Song Of The Year,Bring My Flowers Now,"Brandi Carlile, Phil Hanseroth, Tim Hanseroth & Tanya Tucker, song..."
19,Song Of The Year,Hard Place,"Ruby Amanfu, Sam Ashworth, D. Arcelious Harris, H.E.R. & Rodney Je..."


## 3 · Uniqueness and duplication (candidate keys, duplicate patterns, expectation at source grain)

In [4]:
s = spotify
rep_ids = s[s.track_id.duplicated(keep=False)]
grp = rep_ids.groupby("track_id")
p_s_03 = pd.Series({
    "rows": len(s),
    "distinct_track_id": s.track_id.nunique(),
    "rows_with_repeated_track_id (beyond first)": int(s.track_id.duplicated().sum()),
    "track_ids_that_repeat": grp.ngroups,
    "max_genres_for_one_track_id": int(grp.track_genre.nunique().max()),
    "exact_duplicate_rows (track_id+track_genre)": int(s.duplicated(["track_id","track_genre"]).sum()),
    "exact_duplicate_rows (all columns except csv_index)": int(s.drop(columns="csv_index").duplicated().sum()),
    "repeated_ids_with_different_popularity": int((grp.popularity.nunique() > 1).sum()),
    "repeated_ids_with_different_artists": int((grp.artists.nunique() > 1).sum()),
    "rows_repeating (artists+track_name) beyond first": int(s.duplicated(["artists","track_name"]).sum()),
    "csv_index_is_unique_sequence": bool(s.csv_index.is_unique and (s.csv_index == range(len(s))).all()),
}).to_frame("value")
print("P-S-03 Spotify uniqueness"); display(p_s_03); save(p_s_03, "P-S-03", "uniqueness")

gk = ["year", "category", "nominee", "artist"]
p_g_03 = pd.Series({
    "rows": len(g),
    "exact_duplicate_rows": int(g.duplicated().sum()),
    "duplicate_(year,category,nominee,artist)": int(g.duplicated(gk).sum()),
    "duplicate_(year,category,nominee)": int(g.dropna(subset=["nominee"]).duplicated(["year","category","nominee"]).sum()),
    "rows_where_candidate_key_has_NULL": int(g[gk].isna().any(axis=1).sum()),
}).to_frame("value")
print("P-G-03 Grammy uniqueness"); display(p_g_03); save(p_g_03, "P-G-03", "uniqueness")
print("\nReading: Spotify grain is (track_id, track_genre) -> track_id repeats are EXPECTED across genres; "
      "only (track_id, track_genre) repeats are true duplicates.")

P-S-03 Spotify uniqueness


,value
rows,114000
distinct_track_id,89741
rows_with_repeated_track_id (beyond first),24259
track_ids_that_repeat,16641
max_genres_for_one_track_id,9
exact_duplicate_rows (track_id+track_genre),450
exact_duplicate_rows (all columns except csv_index),450
repeated_ids_with_different_popularity,720
repeated_ids_with_different_artists,0
rows_repeating (artists+track_name) beyond first,32656


P-G-03 Grammy uniqueness


,value
rows,4810
exact_duplicate_rows,0
"duplicate_(year,category,nominee,artist)",0
"duplicate_(year,category,nominee)",0
rows_where_candidate_key_has_NULL,1840



Reading: Spotify grain is (track_id, track_genre) -> track_id repeats are EXPECTED across genres; only (track_id, track_genre) repeats are true duplicates.


## 4 · Categorical content (distinct values, frequencies, unexpected labels, spelling/case variation)

In [5]:
# ---- Spotify ----
genre_counts = s.track_genre.value_counts()
p_s_04 = pd.Series({
    "distinct_genres": s.track_genre.nunique(),
    "rows_per_genre_min": int(genre_counts.min()), "rows_per_genre_max": int(genre_counts.max()),
    "explicit_true_pct": round(s.explicit.mean() * 100, 2),
    "key_values": str(sorted(map(int, s.key.unique()))), "mode_values": str(sorted(map(int, s["mode"].unique()))),
    "time_signature_values": str(sorted(map(int, s.time_signature.unique()))),
    "artists_with_multiple_names_(';')": int(s.artists.fillna("").str.contains(";").sum()),
    "artists_multi_pct": round(s.artists.fillna("").str.contains(";").mean() * 100, 2),
    "artists_with_leading/trailing_space": int((s.artists.fillna("") != s.artists.fillna("").str.strip()).sum()),
}).to_frame("value")
print("P-S-04 Spotify categorical"); display(p_s_04); save(p_s_04, "P-S-04", "categorical")
ind = s.artists.dropna().str.split(";").explode()
print("individual artist names: raw =", ind.nunique(), "| lower+strip =", ind.str.lower().str.strip().nunique())
print("Example trailing-space artists:", s[s.artists.fillna("") != s.artists.fillna("").str.strip()].artists.unique().tolist()[:5])

# ---- Grammy ----
cat = g.category.drop_duplicates()
key = cat.str.lower().str.strip()
variants = cat[key.duplicated(keep=False)].to_frame("category").assign(norm=key[key.duplicated(keep=False)]).sort_values("norm")
p_g_04 = variants.rename(columns={"category": "category_as_delivered", "norm": "lowercase_form"})
print(f"P-G-04 Grammy categories: distinct as delivered = {cat.nunique()}, after lower+strip = {key.nunique()}")
display(p_g_04.head(10)); save(p_g_04, "P-G-04", "category_case_variants", index=False)

p_g_07 = g.winner.value_counts(dropna=False).to_frame("rows")
print("P-G-07 Grammy winner domain"); display(p_g_07); save(p_g_07, "P-G-07", "winner_domain")

p_g_05 = pd.Series({
    "artist_non_null_rows": int(g.artist.notna().sum()),
    "'(Various Artists)' rows": int((g.artist == "(Various Artists)").sum()),
    "artist_composite_(& , ; Featuring)": int(g.artist.fillna("").str.contains(r"&|;|,|featuring", case=False).sum()),
    "artist_with_leading/trailing_space": int((g.artist.fillna("") != g.artist.fillna("").str.strip()).sum()),
    "title_with_double_space": int(g.title.str.contains("  ").sum()),
    "title_example": g.title.iloc[0],
}).to_frame("value")
print("P-G-05 Grammy artist/title content"); display(p_g_05); save(p_g_05, "P-G-05", "artist_title_content")
print("Top Grammy artists:"); display(g.artist.value_counts().head(8).to_frame("rows"))

P-S-04 Spotify categorical


,value
distinct_genres,114
rows_per_genre_min,1000
rows_per_genre_max,1000
explicit_true_pct,8.55
key_values,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11]"
mode_values,"[0, 1]"
time_signature_values,"[0, 1, 3, 4, 5]"
artists_with_multiple_names_(';'),30075
artists_multi_pct,26.38
artists_with_leading/trailing_space,8


individual artist names: raw = 29859 | lower+strip = 29811
Example trailing-space artists: ['Terra ']
P-G-04 Grammy categories: distinct as delivered = 638, after lower+strip = 630


,category_as_delivered,lowercase_form
3498,Best Classical Performance - Instrumental Soloist Or Soloists (Wit...,best classical performance - instrumental soloist or soloists (wit...
3822,Best Classical Performance - Instrumental Soloist OR Soloists (Wit...,best classical performance - instrumental soloist or soloists (wit...
1202,Best Instrumental Soloist Performance (without Orchestra),best instrumental soloist performance (without orchestra)
2367,Best Instrumental Soloist Performance (Without Orchestra),best instrumental soloist performance (without orchestra)
1201,Best Instrumental Soloist(s) Performance (with Orchestra),best instrumental soloist(s) performance (with orchestra)
2366,Best Instrumental Soloist(s) Performance (With Orchestra),best instrumental soloist(s) performance (with orchestra)
1753,Best Small Ensemble Performance (with or without Conductor),best small ensemble performance (with or without conductor)
2369,Best Small Ensemble Performance (With Or Without Conductor),best small ensemble performance (with or without conductor)
1154,"Best Southern, Country, Or Bluegrass Gospel Album","best southern, country, or bluegrass gospel album"
1705,"Best Southern, Country, or Bluegrass Gospel Album","best southern, country, or bluegrass gospel album"


P-G-07 Grammy winner domain


,rows
winner,
True,4810


P-G-05 Grammy artist/title content


,value
artist_non_null_rows,2970
'(Various Artists)' rows,66
"artist_composite_(& , ; Featuring)",510
artist_with_leading/trailing_space,0
title_with_double_space,4810
title_example,62nd Annual GRAMMY Awards (2019)


Top Grammy artists:


,rows
artist,
(Various Artists),66
U2,18
Aretha Franklin,16
Beyoncé,13
Bruce Springsteen,13
Stevie Wonder,13
Ella Fitzgerald,13
Dixie Chicks,12


## 5 · Numerical content (ranges, impossible values, extremes, units)

In [6]:
num_cols = ["popularity","duration_ms","danceability","energy","key","loudness","speechiness",
            "acousticness","instrumentalness","liveness","valence","tempo","time_signature"]
desc = s[num_cols].describe(percentiles=[.01,.5,.99]).T[["min","1%","50%","99%","max"]].round(3)
print("P-S-05 Spotify numeric ranges"); display(desc); save(desc, "P-S-05", "numeric_ranges")

bounded = ["danceability","energy","speechiness","acousticness","instrumentalness","liveness","valence"]
p_s_06 = pd.Series({
    "bounded_features_outside_[0,1]": int(sum(((s[c] < 0) | (s[c] > 1)).sum() for c in bounded)),
    "popularity_outside_[0,100]": int(((s.popularity < 0) | (s.popularity > 100)).sum()),
    "key_outside_[0,11]": int(((s.key < 0) | (s.key > 11)).sum()),
    "mode_not_in_{0,1}": int((~s["mode"].isin([0, 1])).sum()),
    "popularity_equals_0": int((s.popularity == 0).sum()),
    "popularity_equals_0_pct": round((s.popularity == 0).mean() * 100, 2),
    "tempo_equals_0": int((s.tempo == 0).sum()),
    "time_signature_equals_0": int((s.time_signature == 0).sum()),
    "tempo0_and_time_signature0": int(((s.tempo == 0) & (s.time_signature == 0)).sum()),
    "duration_ms_equals_0": int((s.duration_ms == 0).sum()),
    "duration_under_30s": int((s.duration_ms < 30_000).sum()),
    "duration_over_20min": int((s.duration_ms > 1_200_000).sum()),
    "duration_max_minutes": round(s.duration_ms.max() / 60000, 1),
    "loudness_above_0_dB": int((s.loudness > 0).sum()),
    "loudness_min_dB": float(s.loudness.min()),
}).to_frame("value")
print("P-S-06 Spotify implausible / suspicious values"); display(p_s_06); save(p_s_06, "P-S-06", "suspicious_values")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
s.popularity.plot.hist(bins=50, ax=ax[0], color="#4C78A8"); ax[0].set_title("Spotify popularity (0 spike = see P-S-06)"); ax[0].set_xlabel("popularity")
(s.duration_ms / 60000).clip(upper=12).plot.hist(bins=60, ax=ax[1], color="#4C78A8"); ax[1].set_title("Duration, minutes (clipped at 12)"); ax[1].set_xlabel("minutes")
plt.tight_layout(); plt.savefig(FIG / "fig_spotify_distributions.png", dpi=120); plt.close()

P-S-05 Spotify numeric ranges


,min,1%,50%,99%,max
popularity,0.000,0.000,35.000,80.000,100.000
duration_ms,0.000,64652.900,212906.000,530907.660,5237295.000
danceability,0.000,0.134,0.580,0.901,0.985
energy,0.000,0.031,0.685,0.993,1.000
key,0.000,0.000,5.000,11.000,11.000
loudness,-49.531,-27.788,-7.004,-1.634,4.532
speechiness,0.000,0.025,0.049,0.507,0.965
acousticness,0.000,0.000,0.169,0.992,0.996
instrumentalness,0.000,0.000,0.000,0.955,1.000
liveness,0.000,0.041,0.132,0.946,1.000


P-S-06 Spotify implausible / suspicious values


,value
"bounded_features_outside_[0,1]",0.000
"popularity_outside_[0,100]",0.000
"key_outside_[0,11]",0.000
"mode_not_in_{0,1}",0.000
popularity_equals_0,16020.000
popularity_equals_0_pct,14.050
tempo_equals_0,157.000
time_signature_equals_0,163.000
tempo0_and_time_signature0,157.000
duration_ms_equals_0,1.000


## 6 · Temporal content (coverage, parseability, granularity, gaps)

In [7]:
pub = pd.to_datetime(g.published_at, utc=True, errors="coerce")
upd = pd.to_datetime(g.updated_at, utc=True, errors="coerce")
ordinal = g.title.str.extract(r"^(\d+)")[0].astype(float)
p_g_06 = pd.Series({
    "year_min": int(g.year.min()), "year_max": int(g.year.max()), "distinct_years": int(g.year.nunique()),
    "missing_years_in_range": str(sorted(set(range(g.year.min(), g.year.max() + 1)) - set(g.year.unique()))),
    "rows_per_year_min": int(g.year.value_counts().min()), "rows_per_year_max": int(g.year.value_counts().max()),
    "published_at_unparseable": int(pub.isna().sum()), "updated_at_unparseable": int(upd.isna().sum()),
    "updated_before_published": int((upd < pub).sum()),
    "published_at_range": f"{pub.min().date()} .. {pub.max().date()}",
    "distinct_published_at_values": int(g.published_at.nunique()),
    "title_ordinal_vs_year_mismatch_(year-1957 != ordinal)": int(((g.year - 1957) != ordinal).sum()),
}).to_frame("value")
print("P-G-06 Grammy temporal"); display(p_g_06); save(p_g_06, "P-G-06", "temporal")
print("Reading: published_at/updated_at describe WEB-RECORD publication (only %d distinct values), not the award date; "
      "the analytical time axis is `year`." % g.published_at.nunique())
print("Spotify has NO release-date column -> time analysis (AR-03) can only use the Grammy year.")

P-G-06 Grammy temporal


,value
year_min,1958
year_max,2019
distinct_years,62
missing_years_in_range,[]
rows_per_year_min,28
rows_per_year_max,433
published_at_unparseable,0
updated_at_unparseable,0
updated_before_published,0
published_at_range,2017-11-28 .. 2020-05-19


Reading: published_at/updated_at describe WEB-RECORD publication (only 4 distinct values), not the award date; the analytical time axis is `year`.
Spotify has NO release-date column -> time analysis (AR-03) can only use the Grammy year.


## 7 · Cross-source consistency (can the two sources be matched?)

The sources share **no identifier**. Integration can only use names. This section measures how well names match under progressively stronger normalization, and where matching is unsafe. The normalization function here is **exploratory evidence**, not the final transformation rule.

In [8]:
def norm(x):
    """Exploratory normalization: accent-strip, casefold, drop bracketed text, punctuation->space. Keeps non-Latin letters."""
    if pd.isna(x):
        return None
    x = unicodedata.normalize("NFKD", str(x))
    x = "".join(ch for ch in x if not unicodedata.combining(ch))
    x = x.casefold()
    x = re.sub(r"\(.*?\)|\[.*?\]", " ", x)
    x = re.sub(r"[^\w\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x or None

SPLIT = re.compile(r"\s+(?:&|and|featuring|feat\.?|with)\s+|\s*[,;]\s*", flags=re.I)
def split_artists(a):
    return [p for p in SPLIT.split(str(a)) if p and p.strip()]

# Spotify: one row per (track, individual artist)
sp = s.dropna(subset=["artists"]).copy()
sp["artist_ind"] = sp.artists.str.split(";")
sp = sp.explode("artist_ind")
sp["artist_key"] = sp.artist_ind.map(norm)
sp_keys = set(sp.artist_key.dropna())
sp_raw = set(sp.artist_ind.str.strip())

ga = g.dropna(subset=["artist"])
g_distinct = ga.artist.drop_duplicates()
g_raw = set(g_distinct.str.strip())
g_norm = set(g_distinct.map(norm).dropna())
g_split = {norm(p) for a in g_distinct for p in split_artists(a)} - {None}

def rate(found, total):
    return round(len(found) / len(total) * 100, 1)

p_x_01 = pd.DataFrame([
    {"strategy": "exact string (as delivered, trimmed)", "grammy_distinct_artists": len(g_raw), "matched": len(g_raw & sp_raw), "match_pct": rate(g_raw & sp_raw, g_raw)},
    {"strategy": "normalized name (accents, case, punctuation)", "grammy_distinct_artists": len(g_norm), "matched": len(g_norm & sp_keys), "match_pct": rate(g_norm & sp_keys, g_norm)},
    {"strategy": "normalized + split composite Grammy artists", "grammy_distinct_artists": len(g_split), "matched": len(g_split & sp_keys), "match_pct": rate(g_split & sp_keys, g_split)},
])
print("P-X-01 Grammy artist -> Spotify individual artist"); display(p_x_01); save(p_x_01, "P-X-01", "artist_match_strategies", index=False)

# --- unsafe matches under the split strategy: very short names and common words ---
matched_split = sorted(g_split & sp_keys)
short = [m for m in matched_split if len(m) <= 3]
p_x_04 = pd.DataFrame({"matched_key_with_3_or_fewer_chars": short})
print(f"\nP-X-04 Short / ambiguous matched names (<=3 chars): {len(short)} ->", short)
fp_demo = [m for m in matched_split if m in {"the", "her", "eve", "joy", "sam", "mya", "nao", "zoe", "t i"}]
print("Names that are common words or first names (high false-positive risk):", fp_demo)
# Composite Grammy names that ALSO exist intact as one Spotify artist: splitting them would destroy a valid match
composite = [a for a in g_distinct if len(split_artists(a)) > 1]
intact = [a for a in composite if norm(a) in sp_keys]
p_x_08 = pd.DataFrame({"composite_grammy_artist_existing_intact_in_spotify": intact})
print(f"\nP-X-08 Composite Grammy artist strings: {len(composite)}; of these, {len(intact)} exist intact as ONE Spotify artist (splitting would break them):")
print(intact[:10])
save(p_x_08, "P-X-08", "composite_names_existing_whole", index=False)
save(p_x_04, "P-X-04", "ambiguous_short_matches", index=False)

P-X-01 Grammy artist -> Spotify individual artist


,strategy,grammy_distinct_artists,matched,match_pct
0,"exact string (as delivered, trimmed)",1658,517,31.2
1,"normalized name (accents, case, punctuation)",1618,536,33.1
2,normalized + split composite Grammy artists,1809,699,38.6



P-X-04 Short / ambiguous matched names (<=3 chars): 9 -> ['112', 'eve', 'her', 'joy', 'mya', 'nao', 'sam', 't i', 'zoe']
Names that are common words or first names (high false-positive risk): ['eve', 'her', 'joy', 'mya', 'nao', 'sam', 't i', 'zoe']

P-X-08 Composite Grammy artist strings: 480; of these, 19 exist intact as ONE Spotify artist (splitting would break them):
['Tyler, The Creator', 'for KING & COUNTRY', "I'm With Her", 'Doyle Lawson & Quicksilver', 'Jesse & Joy', '(Miles Davis) & Various Artists', 'For King & Country', 'Mumford & Sons', 'Alison Krauss & Union Station', 'Toots & The Maytals']


'P-X-04_ambiguous_short_matches.csv'

In [9]:
# --- Representativeness: does match rate depend on the era? ---
ga2 = ga.assign(artist_key=ga.artist.map(norm), decade=(ga.year // 10) * 10)
ga2["in_spotify"] = ga2.artist_key.isin(sp_keys)
p_x_02 = (ga2.groupby("decade").agg(grammy_rows_with_artist=("in_spotify", "size"), rows_artist_in_spotify=("in_spotify", "sum")))
p_x_02["match_pct"] = (p_x_02.rows_artist_in_spotify / p_x_02.grammy_rows_with_artist * 100).round(1)
print("P-X-02 Row-level artist match rate by Grammy decade (normalized, no splitting)"); display(p_x_02)
save(p_x_02, "P-X-02", "artist_match_by_decade")

rows_decade = g.assign(decade=(g.year // 10) * 10).groupby("decade").agg(rows=("year", "size"), artist_missing=("artist", lambda x: x.isna().sum()))
rows_decade["artist_missing_pct"] = (rows_decade.artist_missing / rows_decade.rows * 100).round(1)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
rows_decade.rows.plot.bar(ax=ax[0], color="#4C78A8"); ax[0].set_title("Grammy rows per decade"); ax[0].set_xlabel("")
p_x_02.match_pct.plot.bar(ax=ax[1], color="#F58518"); ax[1].set_title("% Grammy rows whose artist exists in Spotify"); ax[1].set_xlabel("")
plt.tight_layout(); plt.savefig(FIG / "fig_grammy_coverage_by_decade.png", dpi=120); plt.close()

P-X-02 Row-level artist match rate by Grammy decade (normalized, no splitting)


,grammy_rows_with_artist,rows_artist_in_spotify,match_pct
decade,,,
1950,33,15,45.5
1960,215,93,43.3
1970,253,118,46.6
1980,420,164,39.0
1990,572,255,44.6
2000,742,324,43.7
2010,735,355,48.3


In [10]:
# --- Row-level matching for AR-03: nominee can be a song, an album or a person ---
gm = g.dropna(subset=["artist", "nominee"]).copy()
gm["a_key"] = gm.artist.map(norm); gm["n_key"] = gm.nominee.map(norm)

sp_t = sp.dropna(subset=["track_name"]).assign(t_key=lambda d: d.track_name.map(norm))
sp_a = sp.dropna(subset=["album_name"]).assign(al_key=lambda d: d.album_name.map(norm))
by_track = set(sp_t.artist_key + "|" + sp_t.t_key)
by_album = set(sp_a.artist_key + "|" + sp_a.al_key)
gm["match_track"] = (gm.a_key + "|" + gm.n_key).isin(by_track)
gm["match_album"] = (gm.a_key + "|" + gm.n_key).isin(by_album)
gm["match_any"] = gm.match_track | gm.match_album
p_x_03 = pd.Series({
    "grammy_rows_with_artist_and_nominee": len(gm),
    "matched_on_track_name": int(gm.match_track.sum()),
    "matched_on_album_name": int(gm.match_album.sum()),
    "matched_either": int(gm.match_any.sum()),
    "matched_either_pct": round(gm.match_any.mean() * 100, 1),
    "grammy_rows_total": len(g),
    "matched_either_pct_of_all_grammy_rows": round(gm.match_any.sum() / len(g) * 100, 1),
}).to_frame("value")
print("P-X-03 Grammy (artist, nominee) -> Spotify track/album"); display(p_x_03); save(p_x_03, "P-X-03", "row_level_match")

# Ambiguity: one Grammy (artist, nominee) matching several Spotify track_ids (remasters, live versions, re-releases)
hit = sp_t.assign(k=sp_t.artist_key + "|" + sp_t.t_key)
hit = hit[hit.k.isin(set(gm.a_key + "|" + gm.n_key))]
ambig = hit.groupby("k").track_id.nunique()
print(f"Matched keys: {len(ambig)} | matching >1 distinct Spotify track_id: {(ambig > 1).sum()} | max track_ids for one key: {ambig.max()}")
save(ambig[ambig > 1].sort_values(ascending=False).head(15).to_frame("distinct_track_ids"), "P-X-05", "ambiguous_track_matches")

# --- Feasibility preview for AR-01/AR-02 (NOT a result: names matched without splitting rules, no DQ policy applied) ---
sp["is_grammy_artist"] = sp.artist_key.isin(g_norm)
base = sp.drop_duplicates(["track_id", "artist_key"])
prev = base.groupby("is_grammy_artist").agg(artist_track_pairs=("track_id", "size"), distinct_artists=("artist_key", "nunique"), avg_popularity=("popularity", "mean")).round(2)
print("Feasibility preview (AR-01) - exploratory only"); display(prev); save(prev, "P-X-06", "feasibility_preview_ar01")
gen = base[base.is_grammy_artist].groupby("track_genre").artist_key.nunique().sort_values(ascending=False).head(8).to_frame("grammy_artists")
print("Feasibility preview (AR-02): genres with most matched Grammy artists"); display(gen); save(gen, "P-X-07", "feasibility_preview_ar02")

P-X-03 Grammy (artist, nominee) -> Spotify track/album


,value
grammy_rows_with_artist_and_nominee,2970.0
matched_on_track_name,294.0
matched_on_album_name,298.0
matched_either,444.0
matched_either_pct,14.9
grammy_rows_total,4810.0
matched_either_pct_of_all_grammy_rows,9.2


Matched keys: 181 | matching >1 distinct Spotify track_id: 50 | max track_ids for one key: 9
Feasibility preview (AR-01) - exploratory only


,artist_track_pairs,distinct_artists,avg_popularity
is_grammy_artist,,,
False,116233,29226,33.46
True,7191,536,33.11


Feasibility preview (AR-02): genres with most matched Grammy artists


,grammy_artists
track_genre,
country,53
alt-rock,46
blues,41
disco,39
latin,37
soul,37
bluegrass,36
dance,35


'P-X-07_feasibility_preview_ar02.csv'

## 8 · Quality risk register (evidence → risk → requirement)

Each row below is backed by an evidence table saved in `docs/evidence/profiling/`. Figures in the *evidence* column are computed from the data above, not typed by hand.

In [11]:
risks = [
 ("RISK-01","Grammy / winner","P-G-07", f"winner = True in {int(g.winner.sum())} of {len(g)} rows (100%); no nominees who lost", "Validity / consistency", "AR-01, AR-02, AR-03"),
 ("RISK-02","Grammy / artist","P-G-02, P-G-08, P-G-09", f"{int(g.artist.isna().sum())} missing ({g.artist.isna().mean()*100:.2f}%); Song Of The Year: {int(sotY.artist.isna().sum())} of {len(sotY)} missing; credits column `workers` present in {int((g.artist.isna() & g.workers.notna()).sum())} of the missing rows", "Completeness", "AR-01, AR-02, AR-03"),
 ("RISK-03","Grammy / nominee","P-G-02", f"{int(g.nominee.isna().sum())} rows with missing nominee (Remixer / New Country Artist categories)", "Completeness", "AR-03"),
 ("RISK-04","Spotify / track_id, track_genre","P-S-03", f"{int(s.track_id.duplicated().sum())} repeated track_id rows across genres; {int(s.duplicated(['track_id','track_genre']).sum())} exact duplicates; {int((grp.popularity.nunique() > 1).sum())} ids with different popularity per genre", "Uniqueness", "AR-01, AR-02, AR-03"),
 ("RISK-05","Spotify / artists","P-S-04", f"{int(s.artists.fillna('').str.contains(';').sum())} rows ({s.artists.fillna('').str.contains(';').mean()*100:.1f}%) hold several artists separated by ';'; {int((s.artists.fillna('') != s.artists.fillna('').str.strip()).sum())} with trailing spaces", "Consistency / validity", "AR-01, AR-02"),
 ("RISK-06","Both / artist names (integration key)","P-X-01, P-X-04", f"no shared ID; exact match {p_x_01.match_pct.iloc[0]}% -> normalized {p_x_01.match_pct.iloc[1]}% -> split {p_x_01.match_pct.iloc[2]}%; {len(short)} matches with <=3 chars; {len(composite)} composite Grammy strings, {len(intact)} of them exist intact in Spotify", "Consistency", "AR-01, AR-02, AR-03"),
 ("RISK-07","Both / coverage of Grammy artists in Spotify","P-X-01, P-X-02, P-X-03", f"only {p_x_01.match_pct.iloc[1]}% of distinct Grammy artists exist in Spotify (normalized); row-level match is {p_x_02.match_pct.min()}%-{p_x_02.match_pct.max()}% across decades (no strong era skew); matched Grammy rows overall: {p_x_03.loc['matched_either_pct_of_all_grammy_rows','value']}% at track/album level", "Completeness / representativeness", "AR-01, AR-02, AR-03"),
 ("RISK-08","Spotify / tempo, time_signature, duration_ms, loudness","P-S-06", f"tempo=0 in {int((s.tempo==0).sum())}, time_signature=0 in {int((s.time_signature==0).sum())}, duration_ms=0 in {int((s.duration_ms==0).sum())}, loudness>0 dB in {int((s.loudness>0).sum())}", "Validity", "AR-03"),
 ("RISK-09","Spotify / popularity","P-S-06", f"popularity = 0 in {int((s.popularity==0).sum())} rows ({(s.popularity==0).mean()*100:.1f}%): real zero or missing-data placeholder", "Validity / accuracy", "AR-01"),
 ("RISK-10","Spotify / artists, album_name, track_name","P-S-02", f"one row (csv_index {int(s[s.artists.isna()].csv_index.iloc[0])}) missing artists, album_name and track_name, duration_ms=0", "Completeness", "AR-01, AR-02, AR-03"),
 ("RISK-11","Grammy / category","P-G-04", f"{cat.nunique()} distinct labels vs {key.nunique()} after lower+strip: case-only spelling variants", "Consistency", "AR-03"),
 ("RISK-12","Grammy / year, published_at; Spotify / (no date)","P-G-06", f"published_at has {g.published_at.nunique()} distinct values (web publication, not award date); year is an award label; Spotify has no release date", "Validity / timeliness", "AR-03"),
 ("RISK-13","Grammy / artist = '(Various Artists)'","P-G-05", f"{int((g.artist == '(Various Artists)').sum())} rows whose artist is a placeholder, not an artist", "Validity", "AR-01, AR-02"),
 ("RISK-14","Grammy / nominee semantics","P-X-03", f"nominee is a song, an album or a person depending on category; track-name match {int(gm.match_track.sum())} vs album-name match {int(gm.match_album.sum())} of {len(gm)} rows", "Consistency", "AR-03"),
]
risk_register = pd.DataFrame(risks, columns=["risk_id","dataset_attribute","evidence_id","profiling_evidence","quality_dimension","related_requirement"])
pd.set_option("display.max_colwidth", 140)
display(risk_register)
save(risk_register, "RISKS", "register", index=False)

,risk_id,dataset_attribute,evidence_id,profiling_evidence,quality_dimension,related_requirement
0,RISK-01,Grammy / winner,P-G-07,winner = True in 4810 of 4810 rows (100%); no nominees who lost,Validity / consistency,"AR-01, AR-02, AR-03"
1,RISK-02,Grammy / artist,"P-G-02, P-G-08, P-G-09",1840 missing (38.25%); Song Of The Year: 70 of 70 missing; credits column `workers` present in 1654 of the missing rows,Completeness,"AR-01, AR-02, AR-03"
2,RISK-03,Grammy / nominee,P-G-02,6 rows with missing nominee (Remixer / New Country Artist categories),Completeness,AR-03
3,RISK-04,"Spotify / track_id, track_genre",P-S-03,24259 repeated track_id rows across genres; 450 exact duplicates; 720 ids with different popularity per genre,Uniqueness,"AR-01, AR-02, AR-03"
4,RISK-05,Spotify / artists,P-S-04,30075 rows (26.4%) hold several artists separated by ';'; 8 with trailing spaces,Consistency / validity,"AR-01, AR-02"
5,RISK-06,Both / artist names (integration key),"P-X-01, P-X-04","no shared ID; exact match 31.2% -> normalized 33.1% -> split 38.6%; 9 matches with <=3 chars; 480 composite Grammy strings, 19 of them e...",Consistency,"AR-01, AR-02, AR-03"
6,RISK-07,Both / coverage of Grammy artists in Spotify,"P-X-01, P-X-02, P-X-03",only 33.1% of distinct Grammy artists exist in Spotify (normalized); row-level match is 39.0%-48.3% across decades (no strong era skew);...,Completeness / representativeness,"AR-01, AR-02, AR-03"
7,RISK-08,"Spotify / tempo, time_signature, duration_ms, loudness",P-S-06,"tempo=0 in 157, time_signature=0 in 163, duration_ms=0 in 1, loudness>0 dB in 90",Validity,AR-03
8,RISK-09,Spotify / popularity,P-S-06,popularity = 0 in 16020 rows (14.1%): real zero or missing-data placeholder,Validity / accuracy,AR-01
9,RISK-10,"Spotify / artists, album_name, track_name",P-S-02,"one row (csv_index 65900) missing artists, album_name and track_name, duration_ms=0",Completeness,"AR-01, AR-02, AR-03"


'RISKS_register.csv'

## 9 · Reading guide

- Profiling **did not alter** any source: all manipulations above operate on copies held in memory.
- The next step (workshop §6.4/§6.5) turns these risks into the dimensional model and into explicit quality rules with metrics, thresholds and severities. Thresholds must be justified by the requirement and the observed data, not by whether the current batch passes.